In [131]:
import pandas as pd 
import numpy as np 
import tensorflow as tf 

In [132]:
data = pd.read_csv("train.csv")

In [133]:
data.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [134]:
data.isnull().sum()

Id                 0
MSSubClass         0
MSZoning           0
LotFrontage      259
LotArea            0
                ... 
MoSold             0
YrSold             0
SaleType           0
SaleCondition      0
SalePrice          0
Length: 81, dtype: int64

In [135]:
data.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive

In [136]:
data = data.drop(['Id','Neighborhood'],axis=1)

In [137]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 79 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MSSubClass     1460 non-null   int64  
 1   MSZoning       1460 non-null   object 
 2   LotFrontage    1201 non-null   float64
 3   LotArea        1460 non-null   int64  
 4   Street         1460 non-null   object 
 5   Alley          91 non-null     object 
 6   LotShape       1460 non-null   object 
 7   LandContour    1460 non-null   object 
 8   Utilities      1460 non-null   object 
 9   LotConfig      1460 non-null   object 
 10  LandSlope      1460 non-null   object 
 11  Condition1     1460 non-null   object 
 12  Condition2     1460 non-null   object 
 13  BldgType       1460 non-null   object 
 14  HouseStyle     1460 non-null   object 
 15  OverallQual    1460 non-null   int64  
 16  OverallCond    1460 non-null   int64  
 17  YearBuilt      1460 non-null   int64  
 18  YearRemo

In [138]:
# use xgboost to find with column is the most importance
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
X = data.drop('SalePrice',axis=1)
y = data['SalePrice']

X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [139]:
#Care missing value
import numpy as np

from sklearn.impute import SimpleImputer

#  1. แยก column
num_cols = X_train.select_dtypes(include=np.number).columns # เลือก column ที่เป็น “ตัวเลข
cat_cols = X_train.select_dtypes(exclude=np.number).columns # เลือก column ที่ “ไม่ใช่ตัวเลข” (string / categorical)

#  2. สร้าง imputer แยก
num_imputer = SimpleImputer(strategy='mean') # ถ้ามี missing → เติมค่า “เฉลี่ย”
cat_imputer = SimpleImputer(strategy='most_frequent')  # สำหรับ categorical เติมค่าที่ “เจอบ่อยที่สุด”

#  3. fit เฉพาะ train
num_imputer.fit(X_train[num_cols])
cat_imputer.fit(X_train[cat_cols])

#  4. transform
X_train[num_cols] = num_imputer.transform(X_train[num_cols])
X_train[cat_cols] = cat_imputer.transform(X_train[cat_cols])

#  5. ใช้กับ test ด้วย 
X_test[num_cols] = num_imputer.transform(X_test[num_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

In [140]:
X_train.isnull().sum()

MSSubClass       0
MSZoning         0
LotFrontage      0
LotArea          0
Street           0
                ..
MiscVal          0
MoSold           0
YrSold           0
SaleType         0
SaleCondition    0
Length: 78, dtype: int64

In [141]:
X_train.columns

Index(['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley',
       'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope',
       'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle', 'RoofMatl',
       'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea', 'ExterQual',
       'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
       'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF',
       'TotalBsmtSF', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical',
       '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath',
       'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr',
       'KitchenQual', 'TotRmsAbvGrd', 'Functional', 'Fireplaces',
       'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish',
       'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond', 'PavedDrive',
       'WoodDeckSF', 'Open

In [142]:
# Onehot Encoding string > Number
from sklearn.compose import ColumnTransformer #การ แปลงตัวแปรอิสระ (X) ที่เป็นข้อความ (categorical) ให้กลายเป็น ตัวเลข เพื่อให้โมเดลเข้าใจได้
from sklearn.preprocessing import OneHotEncoder
cat_cols = X_train.select_dtypes(exclude=np.number).columns
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(handle_unknown='ignore'),cat_cols)], remainder='passthrough')
X_train = ct.fit_transform(X_train)
X_test = ct.transform(X_test)

In [143]:
#Standard Scaler 
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
# apply กับ test
X_test = scaler.transform(X_test)

In [144]:
model = XGBRegressor()
model.fit(X_train,y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [145]:
#Predict Test Result
y_pred = model.predict(X_test)

from sklearn.metrics import mean_absolute_error , mean_squared_error

mae = mean_absolute_error(y_test, y_pred)
print(mae)

16953.93766053082


In [146]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(rmse)

26020.768319458864


In [ ]:
# Find feature that have the most weight 
feature_names = ct.get_feature_names_out() # ใช้เพรามี One-hot encoding แตก column ใหม่
importance = pd.Series(model.feature_importances_, index=feature_names)
importance = importance.sort_values(ascending=False)
print(importance.head(20)) # ดูว่าตัวไหนน่าใช้แล้วเอาcolumn กลับไป train ใหม่อีกรอบ

remainder__OverallQual        0.415627
remainder__GarageCars         0.062979
encoder__GarageFinish_Unf     0.053150
encoder__BsmtQual_Ex          0.049319
remainder__GrLivArea          0.034017
encoder__RoofMatl_CompShg     0.030621
encoder__GarageType_Detchd    0.026440
encoder__CentralAir_N         0.024286
encoder__LandContour_Bnk      0.021704
remainder__KitchenAbvGr       0.021634
encoder__Functional_Typ       0.013776
remainder__TotalBsmtSF        0.012272
remainder__2ndFlrSF           0.009536
remainder__BsmtFinSF1         0.008697
encoder__Functional_Min1      0.008487
remainder__Fireplaces         0.008184
encoder__SaleType_New         0.007089
encoder__MSZoning_RM          0.006735
remainder__1stFlrSF           0.006479
encoder__LandSlope_Gtl        0.005972
dtype: float32


In [151]:
# เอา importance มา ใช้ ANN 
selected_feature =  importance.head.index(20)
X_train_selected = X_train_df[selected_feature]
X_test_selected = X_test_df[selected_feature]

AttributeError: 'function' object has no attribute 'index'